# Dobot Experiment Interface

This notebook is the manual interface. The full experiment flow stays in `experiment.py`.

## Config

In [ ]:
import config

print("Dobot IP:", config.DOBOT_IP)
print("Speed ratio:", config.SPEED_RATIO)
print("Trigger DO:", config.TRIGGER_DO_INDEX)
print("Trigger pulse:", config.TRIGGER_PULSE_SECONDS, "s")
print("Step distance:", config.STEP_DISTANCE_MM, "mm")
print("Total distance:", config.TOTAL_DISTANCE_MM, "mm")
print("Loop count:", config.LOOP_REPEAT_COUNT)

## Manual Robot Interface

In [ ]:
from hardware import (
    connect_robot,
    enable_robot,
    has_robot_error,
    move_linear_point,
    return_to_pose,
    send_do_pulse,
    set_robot_speed,
    start_feedback,
    turn_do_off,
)
from time import sleep

dobot = None
feed_thread = None
initial_pose = None

### Connect And Enable

In [ ]:
dobot = connect_robot(config.DOBOT_IP)

if enable_robot(dobot):
    set_robot_speed(dobot, config.SPEED_RATIO)
    feed_thread = start_feedback(dobot)
    print("Robot ready")
else:
    print("Robot not ready")

### Check Error

In [ ]:
has_robot_error(dobot)

### Get Current Pose

In [ ]:
current_pose = dobot.GetCurrentPose()
current_pose

### Send DO Pulse

In [ ]:
do_on_result, do_off_result = send_do_pulse(
    dobot,
    config.TRIGGER_DO_INDEX,
    config.TRIGGER_PULSE_SECONDS,
)

print("DO ON:", do_on_result)
print("DO OFF:", do_off_result)

### Force DO Off

In [ ]:
turn_do_off(dobot, config.TRIGGER_DO_INDEX)

### Save Initial Pose

In [ ]:
initial_pose = dobot.GetCurrentPose()
print("Initial pose saved:", initial_pose)

### Move Forward To End

Moves from the saved initial pose in the `-Y` direction until `TOTAL_DISTANCE_MM` is reached. Each step sends one DO pulse first.

In [ ]:
if initial_pose is None:
    initial_pose = dobot.GetCurrentPose()
    print("Initial pose was empty; saved current pose:", initial_pose)

step_count = int(config.TOTAL_DISTANCE_MM / config.STEP_DISTANCE_MM)
if abs(step_count * config.STEP_DISTANCE_MM - config.TOTAL_DISTANCE_MM) > 1e-9:
    raise ValueError("TOTAL_DISTANCE_MM must be divisible by STEP_DISTANCE_MM")

print("Moving from initial pose:", initial_pose)
print("Step count:", step_count)

for step_index in range(1, step_count + 1):
    do_on_result, do_off_result = send_do_pulse(
        dobot,
        config.TRIGGER_DO_INDEX,
        config.TRIGGER_PULSE_SECONDS,
    )
    print(f"Step {step_index}/{step_count} DO pulse:", do_on_result, do_off_result)

    target_pose = initial_pose.copy()
    target_pose[1] = initial_pose[1] - config.STEP_DISTANCE_MM * step_index
    print(f"Step {step_index}/{step_count} target pose:", target_pose)
    move_linear_point(dobot, target_pose, config.SPEED_RATIO)
    sleep(config.STEP_WAIT_SECONDS)

end_pose = dobot.GetCurrentPose()
print("End current pose:", end_pose)

### Return To Initial Pose

In [ ]:
if initial_pose is None:
    raise RuntimeError("Run 'Save Initial Pose' before returning.")

return_to_pose(
    dobot,
    initial_pose,
    config.SPEED_RATIO,
    do_indexes=[config.TRIGGER_DO_INDEX],
)

current_pose = dobot.GetCurrentPose()
print("Current pose after return:", current_pose)

## Run Full Experiment

This runs the full flow from `experiment.py`.

In [ ]:
from experiment import run_experiment

records = run_experiment(require_confirm=True)

## View Records

In [ ]:
records

In [ ]:
try:
    import pandas as pd
    df = pd.DataFrame(records)
    display(df)
except ImportError:
    print("pandas is not installed; showing raw records instead.")
    records